In [2]:
# Abrimos el archivo con datos limpios
import pandas as pd

df = pd.read_csv(r"C:\Users\Dami\Desktop\Lumi\BOOTCAMP DATA ANALYSIS\SIMULACION\Bank_Marketing_Cleaned.csv")

df.head()

,id,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,deposit
0,1,59,admin.,married,secondary,no,2343,yes,no,unknown,5,5,1042,1,-1,0,unknown,yes
1,2,56,admin.,married,secondary,no,45,no,no,unknown,5,5,1467,1,-1,0,unknown,yes
2,3,41,technician,married,secondary,no,1270,yes,no,unknown,5,5,1389,1,-1,0,unknown,yes
3,4,55,services,married,secondary,no,2476,yes,no,unknown,5,5,579,1,-1,0,unknown,yes
4,5,54,admin.,married,tertiary,no,184,no,no,unknown,5,5,673,2,-1,0,unknown,yes


In [3]:
# Para saber si hay clientes que no han sido contactados

(df["campaign"] == 0).sum()

np.int64(0)

# **Calculos de KPI's**



In [4]:
# Porcentaje de Conversion a Deposito

# Número clientes que suscribieron depósito
num_subscribers = (df["deposit"] == "yes").sum()

# Número total clientes contactados
total_clients = len(df)

# KPI Conversión
conversion_rate = (num_subscribers / total_clients) * 100

print(f"Porcentaje Conversión Depósito: {conversion_rate:.2f}%")

Porcentaje Conversión Depósito: 49.70%


In [5]:
# Para validar que no hayan valores vacios

df[df["deposit"]=="yes"]["duration"].isna().sum()

np.int64(0)

In [6]:
# Promedio de Duracion de Llamadas de Suscriptores

# Filtrar suscriptores
subscribers = df[df["deposit"] == "yes"]

# Suma duración llamadas suscriptores
total_duration = subscribers["duration"].sum()

# Número suscriptores
num_subscribers = len(subscribers)

# KPI promedio duración
avg_duration_subscribers = total_duration / num_subscribers

print(f"Promedio duración llamadas suscriptores: {avg_duration_subscribers:.2f} segundos")

Promedio duración llamadas suscriptores: 537.29 segundos


In [7]:
# Porcentaje de Llamadas a Telefono o Movil

# Número llamadas realizadas por teléfono o móvil
calls_phone_mobile = df["contact"].isin(["telephone", "cellular"]).sum()

# Número total llamadas
total_calls = len(df)

# KPI porcentaje llamadas teléfono o móvil
percentage_calls = round(
    (calls_phone_mobile / total_calls) * 100,
    2
)

print(f"Porcentaje llamadas teléfono o móvil: {percentage_calls}%")

Porcentaje llamadas teléfono o móvil: 79.36%


In [8]:
# Promedio de Contactos Previos a Suscriptores

# Filtrar suscriptores
subscribers = df[df["deposit"] == "yes"]

# Suma contactos previos
total_previous_contacts = subscribers["previous"].sum()

# Número suscriptores
num_subscribers = len(subscribers)

# KPI promedio contactos previos
avg_previous_contacts = total_previous_contacts / num_subscribers

print(f"Promedio contactos previos suscriptores: {avg_previous_contacts:.2f}")

Promedio contactos previos suscriptores: 1.17


In [9]:
# Mes con Mayor Tasa de Conversion

# Agrupar por mes
monthly_kpi = df.groupby("month").agg(

    # Número total clientes contactados en el mes
    total_clientes=("deposit", "count"),

    # Número suscriptores en el mes
    suscriptores=("deposit", lambda x: (x == "yes").sum())

)

# Calcular tasa conversión mensual con redondeo
monthly_kpi["conversion_rate"] = round(
    (monthly_kpi["suscriptores"] /
     monthly_kpi["total_clientes"]) * 100,
    2   # número de decimales
)

print("Conversión por mes:")
print(monthly_kpi)


# Mes con mayor conversión
best_month = monthly_kpi["conversion_rate"].idxmax()

best_value = monthly_kpi["conversion_rate"].max()

print(f"\nMes con mayor tasa de conversión: {best_month} ({best_value:.2f}%)")

Conversión por mes:
       total_clientes  suscriptores  conversion_rate
month                                               
1                 327           142            43.43
2                 746           441            59.12
3                 274           248            90.51
4                 888           577            64.98
5                2646           925            34.96
6                1174           546            46.51
7                1429           627            43.88
8                1450           688            47.45
9                 314           269            85.67
10                387           323            83.46
11                898           403            44.88
12                109           100            91.74

Mes con mayor tasa de conversión: 12 (91.74%)


In [10]:
# %pip install plotly jinja2

In [14]:
import plotly.express as px
import plotly.graph_objects as go

# KPIs por mes
monthly_dashboard = df.groupby("month").agg(

conversion_rate=("deposit",
lambda x:(x=="yes").mean()*100),

avg_duration=("duration","mean"),

avg_previous=("previous","mean"),

phone_mobile_pct=("contact",
lambda x:x.isin(["telephone","cellular"]).mean()*100)

).round(2)

# Selector interactivo
fig = go.Figure()

for month in monthly_dashboard.index:

    fig.add_trace(

        go.Bar(

            x=[
            "Conversion %",
            "Avg Duration",
            "Avg Previous",
            "Phone/Mobile %"
            ],

            y=[

            monthly_dashboard.loc[month,"conversion_rate"],

            monthly_dashboard.loc[month,"avg_duration"],

            monthly_dashboard.loc[month,"avg_previous"],

            monthly_dashboard.loc[month,"phone_mobile_pct"]

            ],

            name=f"Month {month}",

            visible=False

        )

    )

# Mostrar primer mes
fig.data[0].visible=True

# Botones selector mes
buttons=[]

for i,month in enumerate(monthly_dashboard.index):

    visible=[False]*len(fig.data)

    visible[i]=True

    buttons.append(dict(

        label=f"Month {month}",

        method="update",

        args=[{"visible":visible}]

    ))

fig.update_layout(

title="Credit Risk KPI Control Center",

updatemenus=[dict(

buttons=buttons,

direction="down",

showactive=True

)]

)

fig.write_html("control_center_dashboard.html")